In [182]:
import pandas as pd
import numpy as np



In [183]:
df =pd.read_csv("housing.csv")

In [184]:
df["income_cat"]=pd.cut(df["median_income"],bins=[0,1.5,3.0,4.0,6.0,np.inf],labels=[1,2,3,4,5])

In [185]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df["income_cat"]):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

In [186]:
#Removing the Income Category(income_cat) column
for sett in (strat_train_set,strat_test_set):
    sett.drop("income_cat",axis=1 ,inplace=True)
    

In [209]:
df=strat_train_set.copy()

In [210]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
3945,-118.62,34.21,26.0,3234.0,517.0,1597.0,513.0,6.1074,258600.0,<1H OCEAN
7086,-118.00,33.93,35.0,1288.0,240.0,758.0,250.0,4.9205,173900.0,<1H OCEAN
5553,-118.37,33.95,5.0,6955.0,2062.0,3591.0,1566.0,3.1110,247600.0,<1H OCEAN
14437,-117.24,32.80,18.0,2205.0,661.0,874.0,580.0,3.8018,112500.0,NEAR OCEAN
10691,-117.72,33.63,15.0,1362.0,255.0,378.0,202.0,1.9000,162500.0,<1H OCEAN
...,...,...,...,...,...,...,...,...,...,...
6722,-118.14,34.13,49.0,4438.0,803.0,1650.0,741.0,5.1072,479700.0,<1H OCEAN
2760,-115.52,32.67,6.0,2804.0,581.0,2807.0,594.0,2.0625,67700.0,INLAND
2238,-119.82,36.84,7.0,2289.0,342.0,1077.0,354.0,5.4868,158800.0,INLAND
8259,-118.18,33.77,30.0,1418.0,439.0,720.0,417.0,2.6371,159400.0,NEAR OCEAN


In [189]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

In [190]:
from sklearn.impute import SimpleImputer

In [191]:
imputer =SimpleImputer(strategy="median")

In [192]:
housing_num=housing.select_dtypes(include=[np.number])

In [193]:
imputer.fit(housing_num)

,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [194]:
imputer.statistics_

array([-118.495 ,   34.26  ,   29.    , 2121.    ,  432.    , 1164.5   ,
        409.    ,    3.5409])

In [195]:
x=imputer.transform(housing_num)

In [196]:
x

array([[-1.1862e+02,  3.4210e+01,  2.6000e+01, ...,  1.5970e+03,
         5.1300e+02,  6.1074e+00],
       [-1.1800e+02,  3.3930e+01,  3.5000e+01, ...,  7.5800e+02,
         2.5000e+02,  4.9205e+00],
       [-1.1837e+02,  3.3950e+01,  5.0000e+00, ...,  3.5910e+03,
         1.5660e+03,  3.1110e+00],
       ...,
       [-1.1982e+02,  3.6840e+01,  7.0000e+00, ...,  1.0770e+03,
         3.5400e+02,  5.4868e+00],
       [-1.1818e+02,  3.3770e+01,  3.0000e+01, ...,  7.2000e+02,
         4.1700e+02,  2.6371e+00],
       [-1.1844e+02,  3.4180e+01,  3.5000e+01, ...,  5.5000e+02,
         2.5600e+02,  2.2461e+00]], shape=(16512, 8))

In [197]:
housing = pd.DataFrame(x,columns=housing_num.columns, index=housing_num.index)


In [198]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
3945,-118.62,34.21,26.0,3234.0,517.0,1597.0,513.0,6.1074
7086,-118.00,33.93,35.0,1288.0,240.0,758.0,250.0,4.9205
5553,-118.37,33.95,5.0,6955.0,2062.0,3591.0,1566.0,3.1110
14437,-117.24,32.80,18.0,2205.0,661.0,874.0,580.0,3.8018
10691,-117.72,33.63,15.0,1362.0,255.0,378.0,202.0,1.9000
...,...,...,...,...,...,...,...,...
6722,-118.14,34.13,49.0,4438.0,803.0,1650.0,741.0,5.1072
2760,-115.52,32.67,6.0,2804.0,581.0,2807.0,594.0,2.0625
2238,-119.82,36.84,7.0,2289.0,342.0,1077.0,354.0,5.4868
8259,-118.18,33.77,30.0,1418.0,439.0,720.0,417.0,2.6371


In [199]:
housing["ocean_proximity"]=df["ocean_proximity"]


In [200]:
housing =housing[["ocean_proximity"]]

In [201]:
housing

,ocean_proximity
3945,<1H OCEAN
7086,<1H OCEAN
5553,<1H OCEAN
14437,NEAR OCEAN
10691,<1H OCEAN
...,...
6722,<1H OCEAN
2760,INLAND
2238,INLAND
8259,NEAR OCEAN


In [202]:
set(housing["ocean_proximity"])

{'<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'}

In [203]:
from sklearn.preprocessing import OneHotEncoder
ordinal_encoder = OneHotEncoder()
housing_cat= ordinal_encoder.fit_transform(housing)


In [204]:
ordinal_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [207]:
housing_cat.toarray()

array([[1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       ...,
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.]], shape=(16512, 5))

In [212]:
df=pd.concat([df,housing_cat],axis=1)

TypeError: cannot concatenate object of type '<class 'scipy.sparse._csr.csr_matrix'>'; only Series and DataFrame objs are valid

In [ ]:
from sklearn preprocessing